# Mobility-Aid Image Classification — Data Preparation and ResNet-18 Training

This notebook prepares the image-level labels, creates group-based data splits, fine-tunes ResNet-18, and saves the reusable datasets and model checkpoint.

**Target definition:** class `1` means at least one mobility-aid category (`wheelchair`, `walking_frame`, `crutches`, or `push_wheelchair`) appears in the image annotation. Class `0` means the annotation contains other object categories but none of those mobility-aid categories.

Run this notebook when you need to rebuild the data splits or retrain the model.

## 1. Imports, reproducibility and paths

In [2]:
# If needed, install dependencies in a separate notebook cell:
# %pip install pyyaml scikit-learn pandas matplotlib pillow tqdm torch torchvision

import copy
import random
import yaml
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Torchvision:", __import__("torchvision").__version__)

# Expected layout: dataset folders are beside this notebook/project root.
BASE_PATH = Path("./")
IMAGE_PATH = BASE_PATH / "Images_RGB"
ANNOTATION_PATH = BASE_PATH / "Annotations_RGB"

DATA_DIR = BASE_PATH / "data"
MODEL_DIR = BASE_PATH / "models"

for folder in (DATA_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

if not IMAGE_PATH.is_dir() or not ANNOTATION_PATH.is_dir():
    raise FileNotFoundError(
        "Dataset folders not found. Place Images_RGB/ and Annotations_RGB/ "
        "in the project root, or update IMAGE_PATH and ANNOTATION_PATH."
    )

Device: cuda
PyTorch: 2.7.1+cu126
Torchvision: 0.22.1+cu126


## 2. Match images and annotations; create image-level labels

In [3]:
image_files = list(IMAGE_PATH.glob("*.png"))
annotation_files = list(ANNOTATION_PATH.glob("*.yml"))

image_lookup = {p.stem: p for p in image_files}
annotation_lookup = {p.stem: p for p in annotation_files}
common_ids = sorted(set(image_lookup) & set(annotation_lookup))

print("Number of images:", len(image_files))
print("Number of annotation files:", len(annotation_files))
print("Matched pairs:", len(common_ids))
print("Images without annotations:", len(set(image_lookup) - set(annotation_lookup)))
print("Annotations without images:", len(set(annotation_lookup) - set(image_lookup)))

mobility_aid_categories = {
    "wheelchair", "walking_frame", "crutches", "push_wheelchair"
}

records = []
for identifier in common_ids:
    with open(annotation_lookup[identifier], "r", encoding="utf-8") as f:
        content = f.read().replace("%YAML 1.0", "", 1)

    parsed = yaml.safe_load(content) or {}
    annotation = parsed.get("annotation", {})
    objects = annotation.get("object", []) or []

    if isinstance(objects, dict):
        objects = [objects]

    categories = [
        obj["name"] for obj in objects
        if isinstance(obj, dict) and "name" in obj
    ]

    # Empty annotation files are excluded below because they do not establish
    # that the image is a reliable negative example.
    label = int(any(name in mobility_aid_categories for name in categories))

    records.append({
        "image_id": identifier,
        "image_path": str(image_lookup[identifier]),
        "categories": categories,
        "num_objects": len(categories),
        "label": label
    })

df_all_matched = pd.DataFrame(records)
print("Matched annotation rows:", len(df_all_matched))
print("Rows with empty annotations:", int((df_all_matched["num_objects"] == 0).sum()))
print("\nClass counts before excluding empty annotations:")
print(df_all_matched["label"].value_counts().sort_index())

Number of images: 17079
Number of annotation files: 15278
Matched pairs: 15278
Images without annotations: 1801
Annotations without images: 0
Matched annotation rows: 15278
Rows with empty annotations: 27

Class counts before excluding empty annotations:
label
0     1106
1    14172
Name: count, dtype: int64


## 3. Clean the dataset and inspect label distribution

Empty annotation files are excluded rather than treated as confirmed negatives. A negative example therefore has at least one annotated object, but none of the four mobility-aid categories.

In [4]:
df = df_all_matched.loc[df_all_matched["num_objects"] > 0].copy()
df = df.reset_index(drop=True)

print("Cleaned dataset size:", len(df))
print("\nClass counts:")
print(df["label"].value_counts().sort_index())
print("\nClass proportions:")
print(df["label"].value_counts(normalize=True).sort_index().round(4))
print("\nCategory counts:")
print(df["categories"].explode().value_counts())

Cleaned dataset size: 15251

Class counts:
label
0     1079
1    14172
Name: count, dtype: int64

Class proportions:
label
0    0.0707
1    0.9293
Name: proportion, dtype: float64

Category counts:
categories
walking_frame      6725
person             6172
wheelchair         6116
crutches           4685
push_wheelchair    2975
Name: count, dtype: int64


## 4. Create timestamp-based sequence groups

In [5]:
# The filename format is seq_<timestamp>.png.
df["timestamp"] = (
    df["image_id"].str.replace("seq_", "", regex=False).astype(float)
)
df = df.sort_values("timestamp").reset_index(drop=True)
df["time_difference"] = df["timestamp"].diff()

# Heuristic: a gap greater than 30 seconds starts a new timestamp group.
# This is not guaranteed to reproduce the dataset's original recording IDs.
df["sequence_id"] = (df["time_difference"] > 30).cumsum()

print("Number of timestamp-based groups:", df["sequence_id"].nunique())
print("\nLargest groups:")
print(df["sequence_id"].value_counts().head(10))

Number of timestamp-based groups: 34

Largest groups:
sequence_id
16    2606
19    2116
17    2099
32    1484
22     579
12     530
10     411
8      409
25     389
14     383
Name: count, dtype: int64


## 5. Create group-based train, validation and test splits

The groups are kept intact to reduce leakage from near-duplicate consecutive video frames. The split targets are approximately 70% training, 15% validation and 15% test. Since groups cannot be divided, exact proportions are not guaranteed.

In [6]:
overall_ratio = df["label"].mean()

def find_best_group_split(data, test_size, target_ratio, n_trials=2000, random_state=SEED):
    splitter = GroupShuffleSplit(
        n_splits=n_trials,
        test_size=test_size,
        random_state=random_state
    )

    best_split = None
    best_score = float("inf")

    for first_idx, second_idx in splitter.split(data, groups=data["sequence_id"]):
        first = data.iloc[first_idx]
        second = data.iloc[second_idx]

        # Require both classes in both subsets.
        if first["label"].nunique() < 2 or second["label"].nunique() < 2:
            continue

        score = (
            abs(first["label"].mean() - target_ratio)
            + abs(second["label"].mean() - target_ratio)
            + abs(len(first) / len(data) - (1 - test_size))
            + abs(len(second) / len(data) - test_size)
        )

        if score < best_score:
            best_score = score
            best_split = (first_idx, second_idx)

    if best_split is None:
        raise ValueError("No suitable group split found. Try more trials or review group composition.")

    return best_split

# First: approximately 70% train and 30% temporary.
train_idx, temp_idx = find_best_group_split(
    df, test_size=0.30, target_ratio=overall_ratio
)
train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

# Second: split the temporary portion approximately 50/50 into validation and test.
val_idx, test_idx = find_best_group_split(
    temp_df, test_size=0.50, target_ratio=overall_ratio
)
val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# Keep stable row indices for clean exports and downstream use.
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

## 6. Verify and export the data splits

In [7]:
splits = {"train": train_df, "validation": val_df, "test": test_df}

for name, subset in splits.items():
    print(f"\\n{name.title()} set")
    print("Images:", len(subset))
    print("Class counts:")
    print(subset["label"].value_counts().sort_index())
    print("Positive ratio:", round(subset["label"].mean(), 4))
    print("Groups:", subset["sequence_id"].nunique())

train_groups = set(train_df["sequence_id"])
val_groups = set(val_df["sequence_id"])
test_groups = set(test_df["sequence_id"])

print("\\nGroup overlap checks:")
print("Train/validation:", train_groups & val_groups)
print("Train/test:", train_groups & test_groups)
print("Validation/test:", val_groups & test_groups)

assert not (train_groups & val_groups)
assert not (train_groups & test_groups)
assert not (val_groups & test_groups)
assert len(train_df) + len(val_df) + len(test_df) == len(df)

# Save cleaned metadata and exact split membership for reuse.
df.to_csv(DATA_DIR / "cleaned_dataset.csv", index=False)
train_df.to_csv(DATA_DIR / "train.csv", index=False)
val_df.to_csv(DATA_DIR / "validation.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)

print("\\nCSV files saved in:", DATA_DIR.resolve())

\nTrain set
Images: 10650
Class counts:
label
0     754
1    9896
Name: count, dtype: int64
Positive ratio: 0.9292
Groups: 23
\nValidation set
Images: 2649
Class counts:
label
0     201
1    2448
Name: count, dtype: int64
Positive ratio: 0.9241
Groups: 5
\nTest set
Images: 1952
Class counts:
label
0     124
1    1828
Name: count, dtype: int64
Positive ratio: 0.9365
Groups: 6
\nGroup overlap checks:
Train/validation: set()
Train/test: set()
Validation/test: set()
\nCSV files saved in: C:\Users\Ingo\ownCloud - u100358@u100358.ownbackup.at\Jku\AI-VIS\mobility-aids-resnet50\data


## 7. Image transformations and PyTorch Dataset
## 8. Create DataLoaders and check a batch

In [9]:
from src import MobilityAidDataset
from src import train_transform
from src import eval_transform

BATCH_SIZE = 32

train_dataset = MobilityAidDataset(train_df, transform=train_transform)
val_dataset = MobilityAidDataset(val_df, transform=eval_transform)
test_dataset = MobilityAidDataset(test_df, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels in first batch:", labels.tolist())

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels in first batch: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1]


## 9. Configure and fine-tune ResNet-18

In [9]:
# Calculate class weights using TRAINING data only.
class_counts = train_df["label"].value_counts().sort_index()
if len(class_counts) != 2 or (class_counts == 0).any():
    raise ValueError("Training split must contain examples from both classes.")

class_weights = len(train_df) / (2 * class_counts)
class_weights = torch.tensor(class_weights.values, dtype=torch.float32, device=DEVICE)
print("Training class counts:", class_counts.to_dict())
print("Class weights:", class_weights)

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

NUM_EPOCHS = 10
best_val_loss = float("inf")
best_epoch = None
best_model_weights = None

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": []
}

Training class counts: {0: 754, 1: 9896}
Class weights: tensor([7.0623, 0.5381], device='cuda:0')


## 10. Training loop and checkpoint saving

In [10]:
checkpoint_path = MODEL_DIR / "best_resnet18.pth"

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_accuracy = correct / total

    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_accuracy"].append(val_accuracy)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(best_model_weights, checkpoint_path)

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train loss: {train_loss:.4f} | Train acc: {train_accuracy:.4f} | "
        f"Val loss: {val_loss:.4f} | Val acc: {val_accuracy:.4f}"
    )

if best_model_weights is None:
    raise RuntimeError("Training did not produce a model checkpoint.")

model.load_state_dict(best_model_weights)
model.eval()

history_df = pd.DataFrame(history)

print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")
print("Checkpoint saved to:", checkpoint_path.resolve())

Epoch [1/10] Train loss: 0.1021 | Train acc: 0.9597 | Val loss: 0.2011 | Val acc: 0.9268
Epoch [2/10] Train loss: 0.0155 | Train acc: 0.9963 | Val loss: 0.1826 | Val acc: 0.9162
Epoch [3/10] Train loss: 0.0094 | Train acc: 0.9972 | Val loss: 0.2312 | Val acc: 0.9328
Epoch [4/10] Train loss: 0.0059 | Train acc: 0.9979 | Val loss: 0.0910 | Val acc: 0.9690
Epoch [5/10] Train loss: 0.0069 | Train acc: 0.9978 | Val loss: 0.1580 | Val acc: 0.9505
Epoch [6/10] Train loss: 0.0086 | Train acc: 0.9977 | Val loss: 0.1899 | Val acc: 0.9422
Epoch [7/10] Train loss: 0.0146 | Train acc: 0.9954 | Val loss: 0.1203 | Val acc: 0.9702
Epoch [8/10] Train loss: 0.0111 | Train acc: 0.9972 | Val loss: 0.2329 | Val acc: 0.9237
Epoch [9/10] Train loss: 0.0109 | Train acc: 0.9961 | Val loss: 0.1584 | Val acc: 0.9419
Epoch [10/10] Train loss: 0.0039 | Train acc: 0.9991 | Val loss: 0.0793 | Val acc: 0.9717
Best epoch: 10
Best validation loss: 0.0793
Checkpoint saved to: C:\Users\Ingo\ownCloud - u100358@u100358.own

## 11. Outputs created by this notebook

After a successful run, the project should contain:

```text
data/
    cleaned_dataset.csv
    train.csv
    validation.csv
    test.csv

models/
    best_resnet18.pth

```


